# Modular Arithmetic Grokking with TransformerLens

## Overview
Training notebook for a 1-layer transformer on modular addition (a + b mod p).
Based on the ARENA 1.5.2 Grokking notebook structure.

## Model Architecture
- 1-layer transformer using TransformerLens
- Task: Given input [a, b, =], predict (a + b) mod p
- Prime modulus p = 113

## Grokking Phenomenon
The model will first memorize the training set (high train accuracy, low test accuracy),
then later "grok" the underlying algorithm (high test accuracy).

## Cell 1: Setup & Imports

In [ ]:
import math
import random
import os
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
import einops
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import wandb

# TransformerLens imports
from transformer_lens import HookedTransformer, HookedTransformerConfig

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Set seeds for reproducibility (matching original grokking paper: seed=0)
seed = 0
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

print(f"Seeds set to {seed}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda
Seeds set to 0


## Cell 2: Modular Arithmetic Task Definition

In [ ]:
# Prime modulus for modular arithmetic
p = 113

def target_fn(a: int, b: int) -> int:
    """Target function: (a + b) mod p"""
    return (a + b) % p

# Generate all possible input pairs
# Input format: [a, b, p] where p is an "equals" token
# The model predicts (a + b) mod p at the position after '='
# Keep data on CPU - DataLoader will handle transfer to GPU
all_data = torch.tensor([(i, j, p) for i in range(p) for j in range(p)])
labels = torch.tensor([target_fn(i, j) for i, j, _ in all_data])

print(f"Total examples: {len(all_data)} ({p}x{p} = {p*p})")
print(f"Example input: {all_data[0].tolist()} -> label: {labels[0].item()}")
print(f"Another example: {all_data[123].tolist()} -> label: {labels[123].item()}")
print(f"Verification: ({all_data[123][0].item()} + {all_data[123][1].item()}) mod {p} = {target_fn(all_data[123][0].item(), all_data[123][1].item())}")

Total examples: 12769 (113x113 = 12769)
Example input: [0, 0, 113] -> label: 0
Another example: [1, 10, 113] -> label: 11
Verification: (1 + 10) mod 113 = 11


## Cell 3: Dataset Split

In [ ]:
# Train/test split (following original grokking paper exactly)
# Use random.shuffle with seed=0 for deterministic split
frac_train = 0.3

# Generate all pairs as list of tuples (matching original notebook format)
pairs = [(i, j, p) for i in range(p) for j in range(p)]

# Shuffle using random.shuffle (seed was set to 0 in imports cell)
random.seed(seed)  # Reset seed for reproducibility
random.shuffle(pairs)

# Split into train and test
div = int(frac_train * len(pairs))
train_pairs = pairs[:div]
test_pairs = pairs[div:]

# Convert to tensors
train_data = torch.tensor(train_pairs)
train_labels = torch.tensor([target_fn(i, j) for i, j, _ in train_pairs])
test_data = torch.tensor(test_pairs)
test_labels = torch.tensor([target_fn(i, j) for i, j, _ in test_pairs])

# Create boolean index arrays for train/test (used for analysis like in original)
is_train = []
is_test = []
for x in range(p):
    for y in range(p):
        if (x, y, p) in train_pairs:
            is_train.append(True)
            is_test.append(False)
        else:
            is_train.append(False)
            is_test.append(True)
is_train = np.array(is_train)
is_test = np.array(is_test)

# Store indices for checkpoint saving
train_indices = torch.tensor([i for i, pair in enumerate([(x, y, p) for x in range(p) for y in range(p)]) if pair in set(train_pairs)])
test_indices = torch.tensor([i for i, pair in enumerate([(x, y, p) for x in range(p) for y in range(p)]) if pair in set(test_pairs)])

print(f"Train samples: {len(train_data)} ({len(train_data)/len(all_data)*100:.1f}%)")
print(f"Test samples: {len(test_data)} ({len(test_data)/len(all_data)*100:.1f}%)")
print(f"Data split uses random.shuffle with seed={seed} (matching original grokking paper)")

# Create datasets
train_dataset = TensorDataset(train_data, train_labels)
test_dataset = TensorDataset(test_data, test_labels)

Train samples: 3830 (30.0%)
Test samples: 8939 (70.0%)
Data split uses random.shuffle with seed=0 (matching original grokking paper)


## Cell 4: Model Configuration

Following the ARENA 1.5.2 Grokking notebook:
- 1-layer transformer
- d_model = 128
- n_heads = 4
- d_mlp = 512
- No LayerNorm (easier to interpret)

In [ ]:
# Training configuration (matching original grokking paper exactly)
config = {
    # Model architecture (from original Neel Nanda notebook)
    "p": p,
    "n_layers": 1,
    "d_vocab": p + 1,       # 0 to p-1 for numbers, p for '=' token
    "d_model": 128,
    "d_mlp": 4 * 128,       # 512
    "n_heads": 4,
    "d_head": 128 // 4,     # 32
    "n_ctx": 3,             # [a, b, =]
    "act_fn": "relu",
    "normalization_type": None,  # No LayerNorm for interpretability
    
    # Training (matching original grokking paper exactly)
    # Full batch training: batch_size = entire training set
    "batch_size": len(train_data),  # Full batch for grokking (3830)
    "learning_rate": 1e-3,
    "weight_decay": 1.0,     # High weight decay is key for grokking
    "max_epochs": 50000,     # Grokking requires many epochs
    
    # Warmup: LambdaLR with min(step/10, 1) as in original
    "use_lambda_lr_warmup": True,  # Use LambdaLR warmup over first 10 steps
    "warmup_steps": 10,  # For LambdaLR: min(step/10, 1)
    
    # Use high-precision cross entropy (float64) as in original
    "use_high_precision_loss": True,
    
    # Logging
    "log_interval": 100,
    "eval_interval": 100,
    "save_interval": 1000,  # Save checkpoint every 100 epochs for analysis
    
    # Paths
    "wandb_project": "modular-grokking",
}

# Create output directory (using the requested path)
config["output_dir"] = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints"
config["wandb_run_name"] = f"modular_p{p}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

os.makedirs(config["output_dir"], exist_ok=True)

print("Configuration (matching original grokking paper):")
for k, v in config.items():
    print(f"  {k}: {v}")
print(f"\nFull batch training: batch_size = {config['batch_size']} (entire train set)")
print(f"Checkpoints saved every {config['save_interval']} epochs to {config['output_dir']}")

Configuration (matching original grokking paper):
  p: 113
  n_layers: 1
  d_vocab: 114
  d_model: 128
  d_mlp: 512
  n_heads: 4
  d_head: 32
  n_ctx: 3
  act_fn: relu
  normalization_type: None
  batch_size: 3830
  learning_rate: 0.001
  weight_decay: 1.0
  max_epochs: 50000
  use_lambda_lr_warmup: True
  warmup_steps: 10
  use_high_precision_loss: True
  log_interval: 100
  eval_interval: 100
  save_interval: 100
  wandb_project: modular-grokking
  output_dir: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints
  wandb_run_name: modular_p113_20260109_115710

Full batch training: batch_size = 3830 (entire train set)
Checkpoints saved every 100 epochs to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints


## Cell 5: Create Model

In [ ]:
# Create HookedTransformer config
cfg = HookedTransformerConfig(
    n_layers=config["n_layers"],
    d_vocab=config["d_vocab"],
    d_model=config["d_model"],
    d_mlp=config["d_mlp"],
    n_heads=config["n_heads"],
    d_head=config["d_head"],
    n_ctx=config["n_ctx"],
    act_fn=config["act_fn"],
    normalization_type=config["normalization_type"],
    device=device,
)

# Create model
model = HookedTransformer(cfg)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created with {n_params:,} trainable parameters")
print(f"\nModel config:")
print(f"  Layers: {cfg.n_layers}")
print(f"  d_model: {cfg.d_model}")
print(f"  n_heads: {cfg.n_heads}")
print(f"  d_head: {cfg.d_head}")
print(f"  d_mlp: {cfg.d_mlp}")

Model created with 227,442 trainable parameters

Model config:
  Layers: 1
  d_model: 128
  n_heads: 4
  d_head: 32
  d_mlp: 512


## Cell 6: Initialize Data Loaders

In [ ]:
# Create data loaders
# Full batch training with deterministic ordering (no shuffle)
# Data is on CPU, pin_memory=True for efficient GPU transfer
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=False,  # No shuffle for deterministic training
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=len(test_dataset),  # Full batch for eval too
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(f"Train batches per epoch: {len(train_loader)} (full batch = {config['batch_size']} samples)")
print(f"Test batches: {len(test_loader)} (full batch = {len(test_dataset)} samples)")
print(f"Deterministic training: shuffle=False for exact reproducibility")

Train batches per epoch: 1 (full batch = 3830 samples)
Test batches: 1 (full batch = 8939 samples)
Deterministic training: shuffle=False for exact reproducibility


## Cell 7: Import Trainer

In [ ]:
# Import trainer from train.py module
import sys
sys.path.insert(0, '/home/s5e/jrosser.s5e/infusion/caesar_toy/modular')
from modular.train import ModularTrainer, retrain_one_epoch, count_parameters

print("Trainer imported from modular.train module.")

Trainer imported from modular.train module.


## Cell 8: Initialize Wandb

In [ ]:
# Initialize wandb
wandb.init(
    project=config["wandb_project"],
    name=config["wandb_run_name"],
    config=config,
)

# Log model architecture
wandb.watch(model, log="all", log_freq=100)

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Cell 9: Train the Model

Training will show the "grokking" phenomenon:
1. First, train accuracy will go to 100% while test accuracy stays low
2. After many more epochs, test accuracy will suddenly jump to near 100%

In [ ]:
# Create trainer with wandb logging and data for checkpoint saving
trainer = ModularTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    config=config,
    device=device,
    wandb_run=wandb,
    # Pass data for checkpoint saving (enables exact reproduction during infusion)
    train_data=train_data,
    train_labels=train_labels,
    test_data=test_data,
    test_labels=test_labels,
    train_indices=train_indices,
    test_indices=test_indices,
)

print(f"Trainer initialized with data saving enabled")
print(f"  Train data shape: {train_data.shape}")
print(f"  Test data shape: {test_data.shape}")

# Train
trainer.train()

Using LambdaLR warmup scheduler: min(step/10, 1)
Using high-precision cross entropy (float64)
Trainer initialized with data saving enabled
  Train data shape: torch.Size([3830, 3])
  Test data shape: torch.Size([8939, 3])
Starting training for 50000 epochs...
Total steps: 50000


  0%|          | 0/50000 [00:00<?, ?it/s]

  0%|          | 102/50000 [00:09<1:19:02, 10.52it/s]


Step 100: val_loss=6.9645, val_acc=0.57%
Saved checkpoint to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints/checkpoint_epoch_100.pt


  0%|          | 202/50000 [00:18<1:17:12, 10.75it/s]

...

  1%|          | 500/50000 [00:45<1:21:25, 10.13it/s]

  1%|          | 500/50000 [00:45<1:14:57, 11.01it/s]



Step 500: val_loss=22.1503, val_acc=3.20%
Saved checkpoint to /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/modular/checkpoints/checkpoint_epoch_500.pt


KeyboardInterrupt: 

wandb: 
wandb: 🚀 View run modular_p113_20260109_115710 at: 


## Cell 10: Evaluation

In [ ]:
# Load best model
best_checkpoint_path = os.path.join(config["output_dir"], "best_model.pt")
if os.path.exists(best_checkpoint_path):
    checkpoint = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from epoch {checkpoint['epoch']}")
    print(f"Best validation accuracy: {checkpoint['best_val_acc']:.2%}")
    
    # Verify data is saved in checkpoint
    if checkpoint.get("train_data") is not None:
        print(f"Checkpoint contains train data: {checkpoint['train_data'].shape}")
        print(f"Checkpoint contains test data: {checkpoint['test_data'].shape}")

model.eval()

# Final evaluation - move data to device for inference
with torch.no_grad():
    # Test accuracy
    test_data_gpu = test_data.to(device)
    test_labels_gpu = test_labels.to(device)
    logits = model(test_data_gpu)
    preds = logits[:, -1, :-1].argmax(dim=-1)
    test_acc = (preds == test_labels_gpu).float().mean().item()
    
    # Train accuracy  
    train_data_gpu = train_data.to(device)
    train_labels_gpu = train_labels.to(device)
    logits_train = model(train_data_gpu)
    preds_train = logits_train[:, -1, :-1].argmax(dim=-1)
    train_acc = (preds_train == train_labels_gpu).float().mean().item()

print(f"\nFinal Results:")
print(f"  Train accuracy: {train_acc:.2%}")
print(f"  Test accuracy: {test_acc:.2%}")

## Cell 11: Fourier Analysis Setup

The key insight from the grokking paper is that the model learns to use Fourier features.
For modular arithmetic, the model represents numbers using their Fourier components.

In [ ]:
def make_fourier_basis(p: int) -> tuple:
    """
    Creates an orthonormal Fourier basis for functions on Z_p.
    
    The basis consists of:
    - Constant function (frequency 0)
    - cos(2*pi*k*x/p) for k = 1, ..., p//2
    - sin(2*pi*k*x/p) for k = 1, ..., p//2
    
    Returns:
        fourier_basis: [p, p] tensor where fourier_basis[i] is the i-th basis vector
        fourier_basis_names: list of names for each basis vector
    """
    fourier_basis = torch.ones(p, p)
    fourier_basis_names = ["Const"]
    
    for k in range(1, p // 2 + 1):
        # cos and sin components for frequency k
        fourier_basis[2 * k - 1] = torch.cos(2 * torch.pi * torch.arange(p) * k / p)
        fourier_basis[2 * k] = torch.sin(2 * torch.pi * torch.arange(p) * k / p)
        fourier_basis_names.extend([f"cos {k}", f"sin {k}"])
    
    # Normalize to be orthonormal
    fourier_basis /= fourier_basis.norm(dim=1, keepdim=True)
    
    return fourier_basis.to(device), fourier_basis_names

fourier_basis, fourier_basis_names = make_fourier_basis(p)
print(f"Fourier basis shape: {fourier_basis.shape}")
print(f"Basis names (first 10): {fourier_basis_names[:10]}")

# Verify orthonormality
dot_products = fourier_basis @ fourier_basis.T
identity_check = torch.allclose(dot_products, torch.eye(p, device=device), atol=1e-5)
print(f"Orthonormality check: {identity_check}")

In [ ]:
def fft1d(x: torch.Tensor) -> torch.Tensor:
    """
    Project vectors onto the 1D Fourier basis.
    
    Args:
        x: [..., p] tensor
    Returns:
        [..., p] tensor of Fourier coefficients
    """
    return x @ fourier_basis.T

def fft2d(tensor: torch.Tensor) -> torch.Tensor:
    """
    Apply 2D Fourier transform to first two dimensions.
    
    Args:
        tensor: [p, p, ...] tensor
    Returns:
        [p, p, ...] tensor of 2D Fourier coefficients
    """
    return einops.einsum(
        tensor, fourier_basis, fourier_basis,
        "px py ..., i px, j py -> i j ..."
    )

print("Fourier transform functions defined.")

## Cell 12: Analyze Embedding Fourier Components

The model learns to embed numbers using specific Fourier frequencies.
For modular addition, the key frequencies are related to the structure of Z_p.

In [ ]:
# Get the embedding matrix (excluding the '=' token)
W_E = model.W_E[:-1].detach()  # Shape: [p, d_model]
print(f"Embedding matrix shape: {W_E.shape}")

# Project embeddings onto Fourier basis
# W_E_fourier[i, j] = projection of j-th embedding dimension onto i-th Fourier basis vector
W_E_fourier = fourier_basis @ W_E  # Shape: [p, d_model]
print(f"Fourier-transformed embeddings shape: {W_E_fourier.shape}")

# Compute the norm of each Fourier component across all embedding dimensions
# This tells us how much each frequency contributes to the embeddings
fourier_norms = einops.reduce(W_E_fourier.pow(2), "freq d_model -> freq", "sum").sqrt()
print(f"Fourier norms shape: {fourier_norms.shape}")

In [ ]:
# Plot the Fourier spectrum of the embeddings
fig = px.bar(
    x=fourier_basis_names,
    y=fourier_norms.cpu().numpy(),
    title="Fourier Spectrum of Token Embeddings",
    labels={"x": "Fourier Component", "y": "Norm"},
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

# Find the top frequencies
top_k = 10
top_indices = fourier_norms.argsort(descending=True)[:top_k]
print(f"\nTop {top_k} Fourier components in embeddings:")
for i, idx in enumerate(top_indices):
    print(f"  {i+1}. {fourier_basis_names[idx]}: {fourier_norms[idx].item():.4f}")

## Cell 13: Visualize Key Frequencies

The grokked model should show that specific frequencies dominate the embeddings.
These correspond to the structure of the modular arithmetic algorithm.

In [ ]:
# Get key frequencies (those with norm above threshold)
threshold = fourier_norms.mean() + fourier_norms.std()
key_freqs = (fourier_norms > threshold).nonzero().squeeze(-1)

print(f"Key frequencies (norm > {threshold:.4f}):")
for idx in key_freqs:
    print(f"  {fourier_basis_names[idx]}: {fourier_norms[idx].item():.4f}")

# Create a heatmap of Fourier components per embedding dimension
fig = px.imshow(
    W_E_fourier.cpu().numpy(),
    x=[f"d_{i}" for i in range(W_E_fourier.shape[1])],
    y=fourier_basis_names,
    title="Fourier Decomposition of Each Embedding Dimension",
    labels={"x": "Embedding Dimension", "y": "Fourier Component", "color": "Coefficient"},
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
)
fig.update_layout(height=800)
fig.show()

## Cell 14: Analyze 2D Structure in Activations

For the task (a + b) mod p, we can look at how the model processes pairs (a, b).

In [ ]:
# Run model on all data and cache activations
model.eval()
with torch.no_grad():
    # Move all_data to device for inference
    all_data_gpu = all_data.to(device)
    logits, cache = model.run_with_cache(all_data_gpu)
    
# Get the logits at the '=' position (excluding the padding token)
final_logits = logits[:, -1, :-1]  # Shape: [p*p, p]
print(f"Final logits shape: {final_logits.shape}")

# Reshape to 2D grid: [a, b, output]
logits_2d = final_logits.reshape(p, p, p)
print(f"Logits 2D shape: {logits_2d.shape}")

In [ ]:
# Compute 2D Fourier transform of the logits
logits_fourier = fft2d(logits_2d)  # Shape: [p, p, p]
print(f"Logits Fourier shape: {logits_fourier.shape}")

# Compute the total energy at each 2D frequency (summed over output logits)
fourier_energy = einops.reduce(logits_fourier.pow(2), "fa fb out -> fa fb", "sum")
print(f"Fourier energy shape: {fourier_energy.shape}")

# Plot 2D Fourier spectrum
fig = px.imshow(
    fourier_energy.cpu().numpy(),
    x=fourier_basis_names,
    y=fourier_basis_names,
    title="2D Fourier Spectrum of Logits",
    labels={"x": "Freq for b", "y": "Freq for a", "color": "Energy"},
    aspect="equal",
)
fig.update_layout(height=700, width=700)
fig.show()

## Cell 15: Identify Key Frequency Pairs

For modular addition (a + b) mod p, the model uses frequency pairs (k, k) that compute
cos(2*pi*k*(a+b)/p) = cos(2*pi*k*a/p)*cos(2*pi*k*b/p) - sin(2*pi*k*a/p)*sin(2*pi*k*b/p)

In [ ]:
# Find the diagonal elements (same frequency for a and b)
diagonal_energy = torch.diag(fourier_energy)

# Plot diagonal Fourier energies
fig = px.bar(
    x=fourier_basis_names,
    y=diagonal_energy.cpu().numpy(),
    title="Diagonal Fourier Energy (same freq for a and b)",
    labels={"x": "Frequency", "y": "Energy"},
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

# Find top diagonal frequencies
print("\nTop diagonal frequencies (key for modular addition):")
top_diag = diagonal_energy.argsort(descending=True)[:10]
for idx in top_diag:
    print(f"  {fourier_basis_names[idx]}: {diagonal_energy[idx].item():.4f}")

## Cell 16: Summary of Learned Frequencies

In [ ]:
print("="*80)
print("SUMMARY: Fourier Analysis of Grokked Model")
print("="*80)

# Embedding frequencies
print("\n1. EMBEDDING FOURIER COMPONENTS")
print("-" * 40)
embed_threshold = fourier_norms.mean() + 0.5 * fourier_norms.std()
key_embed_freqs = (fourier_norms > embed_threshold).nonzero().squeeze(-1)
for idx in key_embed_freqs.tolist():
    print(f"   {fourier_basis_names[idx]}: norm = {fourier_norms[idx].item():.4f}")

# Key 2D frequencies
print("\n2. KEY 2D LOGIT FREQUENCIES")
print("-" * 40)
logit_threshold = fourier_energy.mean() + fourier_energy.std()
key_2d_freqs = (fourier_energy > logit_threshold).nonzero()
for idx in key_2d_freqs:
    i, j = idx[0].item(), idx[1].item()
    print(f"   ({fourier_basis_names[i]}, {fourier_basis_names[j]}): energy = {fourier_energy[i, j].item():.4f}")

# Interpretation
print("\n3. INTERPRETATION")
print("-" * 40)
print("   The model has learned to use specific Fourier frequencies to compute")
print("   (a + b) mod p. The key frequencies form the algorithm:")
print("   ")
print("   For each key frequency k:")
print("     cos(2*pi*k*(a+b)/p) = cos(a)*cos(b) - sin(a)*sin(b)")
print("   ")
print("   This is exactly the angle addition formula, which the model")
print("   implements through its attention and MLP layers.")

## Cell 17: Finish and Cleanup

In [ ]:
# Log final summary to wandb
wandb.summary["final_train_acc"] = train_acc
wandb.summary["final_test_acc"] = test_acc
wandb.summary["best_val_acc"] = trainer.best_val_acc
wandb.summary["total_steps"] = trainer.global_step
wandb.summary["key_frequencies"] = [fourier_basis_names[i] for i in key_embed_freqs.tolist()]

# Finish wandb run
wandb.finish()

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Final train accuracy: {train_acc:.2%}")
print(f"Final test accuracy: {test_acc:.2%}")
print(f"Checkpoints saved to: {config['output_dir']}")
print(f"Wandb run: {config['wandb_run_name']}")